In [1]:
"""
Airbnb Listings Scraper (Selenium + BeautifulSoup)
---------------------------------------------------
Scrapes publicly visible listing data from Airbnb search results pages
(title, price, rating, review count, room type, location snippet, listing URL)
and saves everything to an Excel file.

WHY SELENIUM + BEAUTIFULSOUP:
Airbnb's search results are rendered with JavaScript, so a plain
requests.get() call won't show you the listing cards. Selenium drives a
real (headless) Chrome browser to load and scroll the page so the JS
renders, then BeautifulSoup parses the resulting HTML.

BEFORE YOU RUN THIS:
1. Install Google Chrome on your machine (if not already installed).
2. Install the Python packages:
   pip install selenium beautifulsoup4 pandas openpyxl webdriver-manager

3. Edit the CONFIG section below (search location, number of pages, etc.)

ETHICAL / LEGAL NOTE FOR YOUR PROJECT WRITE-UP:
- This script only reads publicly visible search-result HTML — no login,
  no bypassing paywalls or CAPTCHAs, no private data.
- It rate-limits requests (random delays) to avoid hammering Airbnb's
  servers, which is both good etiquette and reduces the chance of
  getting blocked.
- Airbnb's Terms of Service restrict automated scraping. For a university
  project this is normally fine under fair-use/academic-research
  justification, but you should say explicitly in your methodology
  section that: (a) you only collected publicly available fields relevant
  to your research questions, (b) you rate-limited your requests, and
  (c) no personal host/guest data was collected. This is exactly the kind
  of transparency your marker flagged as missing in the review-of-methods
  feedback.
- If Airbnb changes their page layout, the CSS selectors below WILL need
  updating — that's normal for web scraping and worth noting as a
  limitation in your report.
"""

import time
import random
import re
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service


# ============================================================
# CONFIG — edit these to suit your project
# ============================================================
# Full list of all 33 London boroughs (official Inner/Outer classification).
# Defined first since LOCATIONS below now points at it - the professor
# confirmed more boroughs (not more pages on a few) is what gets us to
# 3,000-4,000+ listings.
FULL_LOCATIONS = [
    "Barking and Dagenham, London, United Kingdom", "Barnet, London, United Kingdom",
    "Bexley, London, United Kingdom", "Brent, London, United Kingdom",
    "Bromley, London, United Kingdom", "Camden, London, United Kingdom",
    "Croydon, London, United Kingdom", "Ealing, London, United Kingdom",
    "Enfield, London, United Kingdom", "Greenwich, London, United Kingdom",
    "Hackney, London, United Kingdom", "Hammersmith and Fulham, London, United Kingdom",
    "Haringey, London, United Kingdom", "Harrow, London, United Kingdom",
    "Havering, London, United Kingdom", "Hillingdon, London, United Kingdom",
    "Hounslow, London, United Kingdom", "Islington, London, United Kingdom",
    "Kensington and Chelsea, London, United Kingdom", "Kingston upon Thames, London, United Kingdom",
    "Lambeth, London, United Kingdom", "Lewisham, London, United Kingdom",
    "Merton, London, United Kingdom", "Newham, London, United Kingdom",
    "Redbridge, London, United Kingdom", "Richmond upon Thames, London, United Kingdom",
    "Southwark, London, United Kingdom", "Sutton, London, United Kingdom",
    "Tower Hamlets, London, United Kingdom", "Waltham Forest, London, United Kingdom",
    "Wandsworth, London, United Kingdom", "Westminster, London, United Kingdom",
    "City of London, United Kingdom",
]

# 16-borough / 2-page test batch used previously (kept here for reference -
# switch LOCATIONS back to this if you want a smaller run for testing).
TEST_BATCH_LOCATIONS = [
    "Camden, London, United Kingdom", "Hackney, London, United Kingdom",
    "Westminster, London, United Kingdom", "Croydon, London, United Kingdom",
    "Ealing, London, United Kingdom", "Greenwich, London, United Kingdom",
    "Islington, London, United Kingdom", "Tower Hamlets, London, United Kingdom",
    "Southwark, London, United Kingdom", "Lambeth, London, United Kingdom",
    "Wandsworth, London, United Kingdom", "Bromley, London, United Kingdom",
    "Barnet, London, United Kingdom", "Enfield, London, United Kingdom",
    "Redbridge, London, United Kingdom", "Sutton, London, United Kingdom",
]

# CURRENT SETTING: test batch (16 boroughs x 2 pages), per professor's
# request to see 300-400 listings from a wider set of boroughs before
# committing to the full run. Switch LOCATIONS to FULL_LOCATIONS and
# PAGES_PER_LOCATION to 5 for the full run, once approved - at that
# setting (33 boroughs x 5 pages x ~20 listings/page) it comfortably
# crosses 4,000 before dedup.
LOCATIONS = TEST_BATCH_LOCATIONS
PAGES_PER_LOCATION = 2

# IMPORTANT: fixed dates are required so every listing quotes the SAME
# number of nights. Without this, Airbnb assigns each card a random
# check-in/check-out window, and the total price becomes incomparable
# across rows (a 4-night £400 stay vs a 10-night £1000 stay).
CHECKIN = "2026-09-01"
CHECKOUT = "2026-09-05"               # 4 nights - change both together if you want a different length
EXPECTED_NIGHTS = (
    datetime.strptime(CHECKOUT, "%Y-%m-%d") - datetime.strptime(CHECKIN, "%Y-%m-%d")
).days
HEADLESS = True                       # False = watch the browser work (good for debugging)
OUTPUT_FILE = f"airbnb_london_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
MIN_DELAY, MAX_DELAY = 3, 7           # seconds between page loads (be polite)

# Full official Inner/Outer London classification (33 boroughs incl. City
# of London), so zone_for_location() never returns None for any borough in
# FULL_LOCATIONS above.
BOROUGH_ZONE_MAP = {
    # Inner London
    "Camden": "Inner London",
    "Islington": "Inner London",
    "Hackney": "Inner London",
    "Westminster": "Inner London",
    "Tower Hamlets": "Inner London",
    "Southwark": "Inner London",
    "Lambeth": "Inner London",
    "Wandsworth": "Inner London",
    "Hammersmith and Fulham": "Inner London",
    "Kensington and Chelsea": "Inner London",
    "Lewisham": "Inner London",
    "Greenwich": "Inner London",
    "City of London": "Inner London",
    # Outer London
    "Croydon": "Outer London",
    "Bromley": "Outer London",
    "Barnet": "Outer London",
    "Ealing": "Outer London",
    "Enfield": "Outer London",
    "Havering": "Outer London",
    "Hillingdon": "Outer London",
    "Redbridge": "Outer London",
    "Sutton": "Outer London",
    "Bexley": "Outer London",
    "Barking and Dagenham": "Outer London",
    "Brent": "Outer London",
    "Harrow": "Outer London",
    "Hounslow": "Outer London",
    "Kingston upon Thames": "Outer London",
    "Merton": "Outer London",
    "Richmond upon Thames": "Outer London",
    "Waltham Forest": "Outer London",
    "Haringey": "Outer London",
    "Newham": "Outer London",
}


def build_search_url(location, checkin=None, checkout=None):
    """
    Builds an Airbnb search URL for the FIRST page of results only.

    BUG FIXED: pagination used to be done via an items_offset URL param,
    but Airbnb's current search backend ignores it - both "page 1" and
    "page 2" requests returned the same page 1 results, which is why 866
    scraped rows only contained 310 unique listings. Subsequent pages are
    now reached by clicking the actual "Next" button in the browser (see
    scrape_one_location), not by building a different URL.
    """
    base = "https://www.airbnb.co.uk/s/{}/homes".format(location.replace(", ", "--").replace(" ", "-"))
    params = []
    if checkin:
        params.append(f"checkin={checkin}")
    if checkout:
        params.append(f"checkout={checkout}")
    return base + ("?" + "&".join(params) if params else "")


def start_driver(headless=True):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,1000")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver


def first_room_id_from_cards(cards):
    """Grabs the numeric room ID of the first card - used to detect whether
    clicking 'Next' actually advanced to a new page, or silently reloaded
    the same one (the exact failure mode that caused the 866-vs-310 gap)."""
    for card in cards:
        link = card.select_one("a[href*='/rooms/']")
        if link:
            m = re.search(r"/rooms/(\d+)", link["href"])
            if m:
                return m.group(1)
    return None


def scroll_page(driver, pause=1.5, scrolls=6):
    """Airbnb lazy-loads cards as you scroll — scroll down in steps to force them in."""
    last_height = driver.execute_script("return document.body.scrollHeight")
    for _ in range(scrolls):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height


def parse_total_price(raw_text):
    """
    Extracts the numeric TOTAL stay price - i.e. what a guest actually pays.

    BUG FIXED: Airbnb shows a struck-through original price next to the
    discounted price for any listing with an active discount, e.g.
    "£870 £751 total". The old version took the FIRST number in the
    string with a bare regex, which is always the higher, pre-discount
    figure - wrong on every discounted row (81% of the batch had this).

    Fix: anchor specifically to the number immediately preceding the word
    "total". Only fall back to "first number found" if no "total" marker
    is present at all (rare, but keeps this from returning nothing).
    """
    if not raw_text:
        return None

    # Primary: the number right before "total". Allows an optional decimal
    # portion (e.g. "£751.00 total") - without this, a regex of [\d,]+ alone
    # matches only the "00" after the decimal point and returns 0.0.
    m = re.search(r"([\d,]+(?:\.\d+)?)\s*total", raw_text, re.IGNORECASE)
    if m:
        return float(m.group(1).replace(",", ""))

    # Fallback: no "total" marker found - take the LAST number in the
    # string rather than the first, since discounted price (if present)
    # is listed second/last, immediately before any trailing text.
    numbers = re.findall(r"[\d,]+", raw_text.replace("£", ""))
    if numbers:
        return float(numbers[-1].replace(",", ""))
    return None


def parse_nights(full_text):
    """
    Looks for a 'for X night(s)' phrase in the card text (Airbnb includes
    this alongside the total price, e.g. '£420 for 4 nights'). Returns None
    if not found, in which case we fall back to EXPECTED_NIGHTS from config.
    """
    m = re.search(r"for\s+(\d+)\s+nights?", full_text, re.IGNORECASE)
    return int(m.group(1)) if m else None


# Known hotel/aparthotel brands that show up in Airbnb search results
# alongside peer-to-peer host listings. Hedonic pricing models (Rosen,
# and the Airbnb-specific literature that follows it) are built around
# peer-to-peer listings - a commercial hotel product priced by different
# logic sitting in the same regression can distort the coefficients, so
# these need to be flagged/filterable rather than silently included.
HOTEL_BRAND_KEYWORDS = [
    "point a", "easyhotel", "staycity", "roomzzz", "premier inn",
    "travelodge", "holiday inn", "ibis", "hilton", "marriott",
    "novotel", "mercure", "hampton by hilton", "citizenm", "z hotel",
    "the z hotel", "aparthotel", "apart hotel", "premier suites",
    "cove aparthotels", "locke", "yotel",
]

# Standard category buckets, checked in this order (first match wins).
# Keeping this to a handful of broad categories - rather than trying to
# reproduce Airbnb's exact wording - makes it usable as a regression
# variable and far more robust to markup/phrasing changes than exact
# phrase matching was.
PROPERTY_TYPE_KEYWORDS = [
    ("Hotel", HOTEL_BRAND_KEYWORDS + ["hotel room", "room in hotel", "hotel"]),
    ("Room", ["private room", "shared room", "room in "]),
    ("Flat", ["rental unit", "apartment", "flat", "condo", "loft", "serviced apartment"]),
    ("House", ["home", "house", "townhouse", "cottage", "bungalow", "villa"]),
]


def classify_property_type(full_text, aria_label):
    """
    Classifies each listing into a standard bucket: Room, Flat, House,
    Hotel, or Other/Unknown. This does NOT search the listing title - the
    title is the host's chosen nickname (e.g. "Cozy flat near Camden
    Market"), not a structured type description, which is why searching it
    previously returned blank for most rows. Instead we search the card's
    aria-label (accessibility text, which tends to carry Airbnb's actual
    type wording, e.g. "Entire rental unit in...") plus the full
    concatenated card text as a fallback.

    Order matters: Hotel is checked first so a hotel brand name doesn't
    get miscategorised as "Room" or "Flat" just because its listing also
    contains those words.
    """
    haystack = f"{aria_label or ''} {full_text or ''}".lower()

    for category, keywords in PROPERTY_TYPE_KEYWORDS:
        if any(kw in haystack for kw in keywords):
            is_hotel = category == "Hotel"
            return category, is_hotel

    return "Other/Unknown", False


def parse_listing_card(card):
    """
    Pulls the fields we care about out of one listing card's HTML.

    KEY FIX vs the previous version: we build ONE concatenated text blob
    (`full_text`) from the entire card first, and run our regexes against
    that blob rather than against a single BeautifulSoup text node. Airbnb
    frequently splits the rating and the review count into two separate
    text nodes in the DOM (e.g. "4.85" in one span, "(32 reviews)" in a
    sibling span) - searching only one node was why review_count came back
    empty even though rating was being found correctly.
    """
    data = {}
    full_text = card.get_text(" ", strip=True)

    # Title / name
    title_tag = card.select_one('[data-testid="listing-card-title"]')
    data["title"] = title_tag.get_text(strip=True) if title_tag else None

    # Price - grab the raw text, then split into total price and nights
    price_tag = card.select_one('span[class*="price"], div[data-testid*="price"]')
    price_text = price_tag.get_text(" ", strip=True) if price_tag else full_text
    data["price_raw"] = price_text

    total_price = parse_total_price(price_text)
    nights = parse_nights(full_text) or parse_nights(price_text)
    data["nights"] = nights if nights else EXPECTED_NIGHTS
    data["nights_source"] = "parsed_from_card" if nights else "assumed_from_search_dates"
    data["price_gbp_total"] = total_price
    data["price_gbp_per_night"] = (
        round(total_price / data["nights"], 2) if total_price and data["nights"] else None
    )

    # Rating + review count.
    # BUG FIXED: the old version searched the whole concatenated card text
    # for "any decimal number followed by an integer", which happily
    # matched bedroom counts, distances, and price fragments on cards with
    # no rating element at all (44 rows came back with impossible <3.0
    # ratings, including one 1.5). Fix: anchor specifically to an element
    # whose aria-label describes the rating (Airbnb's standard pattern is
    # "4.85 out of 5 average rating, 32 reviews"), and only accept a value
    # in the valid 1.0-5.0 range.
    # BUG FIXED: the old version required the rating AND the review count to
    # appear inside the SAME aria-label string (e.g. "4.85 out of 5 average
    # rating, 32 reviews"). In practice Airbnb frequently splits these apart
    # - the rating + "out of 5" sits in one aria-label (screen-reader-only
    # text, which is why searching full_text for "out of 5" as a fallback
    # almost never works either), while the review count is a separate
    # plain-text node like "(32)" with no aria-label at all. When that
    # happens the old code found the rating but silently left review_count
    # as None. Fix: find each one independently across every aria-label on
    # the card plus the visible text, instead of requiring both in one string.
    aria_texts = [el.get("aria-label") for el in card.find_all(attrs={"aria-label": True})]
    aria_texts = [t for t in aria_texts if t]

    data["rating"] = None
    data["review_count"] = None

    # Rating: "X out of 5" lives almost exclusively in aria-labels.
    for src in aria_texts + [full_text]:
        m = re.search(r"(\d\.\d{1,2})\s*out of 5", src, re.IGNORECASE)
        if m and 1.0 <= float(m.group(1)) <= 5.0:
            data["rating"] = float(m.group(1))
            break

    # Review count: search independently, "N reviews" first, then a bare
    # "(N)" as a fallback, across the same set of sources.
    for src in aria_texts + [full_text]:
        m = re.search(r"(\d+)\s*reviews?", src, re.IGNORECASE)
        if m:
            data["review_count"] = int(m.group(1))
            break
    if data["review_count"] is None:
        for src in aria_texts + [full_text]:
            m = re.search(r"\((\d+)\)", src)
            if m:
                data["review_count"] = int(m.group(1))
                break

    # Missing rating/reviews are usually genuinely new listings rather than
    # a parsing failure - flag them explicitly so this is a documented,
    # deliberate modelling choice (e.g. imputation) rather than a silent gap.
    data["is_new_listing"] = data["rating"] is None and data["review_count"] is None

    # Listing URL + aria-label (accessibility text often carries the
    # property-type phrase that the visible subtitle element doesn't
    # reliably expose - see detect_property_type() docstring)
    link_tag = card.select_one("a[href*='/rooms/']")
    data["url"] = ("https://www.airbnb.co.uk" + link_tag["href"]) if link_tag else None
    aria_label = link_tag.get("aria-label") if link_tag else None

    # Clean numeric room ID, stripped of query string. Each duplicate card
    # carries a different tracking ID (e.g. source_impression_id) in its
    # URL query string, so two rows for the SAME listing look like
    # different URLs under a plain URL-based dedup. room_id fixes this.
    room_id_match = re.search(r"/rooms/(\d+)", data["url"] or "")
    data["room_id"] = room_id_match.group(1) if room_id_match else None

    # Property type / hotel flag - standard bucket classification
    data["property_type"], data["is_hotel_brand"] = classify_property_type(full_text, aria_label)

    return data


def zone_for_location(location):
    """Maps a search location string to Inner/Outer London using BOROUGH_ZONE_MAP."""
    for borough, zone in BOROUGH_ZONE_MAP.items():
        if borough.lower() in location.lower():
            return zone
    return None


def go_to_next_page(driver, page, max_pages):
    """
    Clicks the 'Next' button and BLOCKS until the next page has actually
    finished rendering, instead of trusting a fixed sleep before the next
    loop iteration re-reads the page.

    ROOT CAUSE OF THE page_scraped==1 BUG: Airbnb's pagination re-renders
    the results list client-side (the URL and driver readyState never
    change - only the DOM node identity does). The old code clicked
    'Next' and then just slept MIN_DELAY-MAX_DELAY seconds before looping
    back to the top, where it re-queried the page. On a slow render, that
    re-query either (a) found the button missing/misclicked and silently
    fell through, or (b) ran the first-listing-ID guard against a page
    that hadn't swapped its cards out yet, saw the SAME first room ID as
    the previous page, and broke out of the loop - both leave every row
    stamped page_scraped == 1.

    FIX: two-stage explicit wait, with the outcome of each stage printed
    so a pagination failure is visible in the console rather than only
    showing up later as a suspicious column value.
      1. staleness_of the current first card - confirms the OLD cards
         were actually torn down by the click (proves the click landed).
      2. presence_of a NEW listing-card-title - confirms the next page's
         cards have actually mounted (proves content arrived, not just
         that the old content left).
    Returns True if the next page is ready to be read, False if pagination
    should stop for this location.
    """
    try:
        next_btn = driver.find_element(
            By.CSS_SELECTOR,
            'a[aria-label="Next"], button[aria-label="Next"], '
            'a[aria-label="Next page"], nav[aria-label*="pagination" i] a:last-child'
        )
    except Exception:
        print("    PAGINATION: 'Next' button not found in DOM - reached the last "
              "available page for this location.")
        return False

    try:
        old_first_card = driver.find_element(By.CSS_SELECTOR, '[data-testid="listing-card-title"]')
    except Exception:
        old_first_card = None

    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_btn)
    time.sleep(0.5)

    try:
        next_btn.click()
        print(f"  [Page {page + 2}/{max_pages}] PAGINATION: 'Next' button found and clicked.")
    except Exception as e:
        print(f"    PAGINATION: click on 'Next' button failed ({e}) - stopping this location.")
        return False

    if old_first_card is not None:
        try:
            WebDriverWait(driver, 15).until(EC.staleness_of(old_first_card))
            print("    PAGINATION: previous page's cards went stale - re-render confirmed.")
        except Exception:
            print("    PAGINATION WARNING: previous page's cards never went stale after 15s "
                  "- the click may not have registered. Proceeding to check for new cards anyway.")

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="listing-card-title"]'))
        )
        print("    PAGINATION: new listing cards detected on the next page.")
    except Exception:
        print("    PAGINATION: no new cards appeared after clicking 'Next' - stopping this location.")
        return False

    return True


def scrape_one_location(driver, location, max_pages, checkin=None, checkout=None):
    listings = []
    url = build_search_url(location, checkin, checkout)
    print(f"  [Page 1/{max_pages}] Loading: {url}")
    driver.get(url)

    prev_first_room_id = None

    for page in range(max_pages):
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="listing-card-title"]'))
            )
        except Exception:
            print("    No listing cards found on this page — stopping this location "
                  "(may be end of results, or Airbnb blocked/changed layout).")
            break

        scroll_page(driver)
        soup = BeautifulSoup(driver.page_source, "html.parser")

        cards = soup.select('[itemprop="itemListElement"], div[data-testid="card-container"]')
        if not cards:
            cards = soup.select('div:has([data-testid="listing-card-title"])')

        print(f"    Found {len(cards)} listing cards")

        # Pagination sanity check: if the first listing on this page is
        # identical to the first listing on the previous page, "Next"
        # didn't actually move us forward - stop rather than keep
        # re-scraping the same page under a different page number.
        first_room_id = first_room_id_from_cards(cards)
        if page > 0 and first_room_id is not None and first_room_id == prev_first_room_id:
            print("    WARNING: first listing on this page matches the previous page - "
                  "pagination did not advance. Stopping this location to avoid duplicate scraping.")
            break
        prev_first_room_id = first_room_id

        for card in cards:
            listing = parse_listing_card(card)
            listing["search_location"] = location
            listing["zone"] = zone_for_location(location)
            listing["page_scraped"] = page + 1
            listing["scraped_at"] = datetime.now().isoformat(timespec="seconds")
            listings.append(listing)

        if page < max_pages - 1:
            advanced = go_to_next_page(driver, page, max_pages)
            if not advanced:
                break

        delay = random.uniform(MIN_DELAY, MAX_DELAY)
        print(f"    Waiting {delay:.1f}s...")
        time.sleep(delay)

    return listings


def scrape_airbnb(locations, pages_per_location, checkin=None, checkout=None, headless=True):
    driver = start_driver(headless=headless)
    all_listings = []

    try:
        for location in locations:
            print(f"\n=== Scraping: {location} ===")
            listings = scrape_one_location(driver, location, pages_per_location, checkin, checkout)
            print(f"  -> {len(listings)} listings from {location}")
            all_listings.extend(listings)
    finally:
        driver.quit()

    return all_listings


def validate_output(df):
    """
    Prints a summary of the exact things that went wrong last time, so a
    stale file or silent regression gets caught here rather than being
    discovered by whoever reads the spreadsheet next.
    """
    print("\n=== Output validation summary ===")

    if "zone" in df.columns:
        blank_zones = df[df["zone"].isna()]
        if not blank_zones.empty:
            missing_locations = blank_zones["search_location"].unique().tolist()
            print(f"  WARNING: {len(blank_zones)} rows have a BLANK zone. "
                  f"Locations affected: {missing_locations}. "
                  f"Check BOROUGH_ZONE_MAP covers these before sending this file on.")
        else:
            print(f"  OK: zone is populated for all {len(df)} rows.")

    if "rating" in df.columns:
        missing_rating = df["rating"].isna().sum()
        pct = 100 * missing_rating / len(df) if len(df) else 0
        print(f"  {missing_rating}/{len(df)} rows ({pct:.1f}%) have no rating/review_count "
              f"- check 'is_new_listing' flag; decide on imputation vs exclusion before modelling.")

    if "nights_source" in df.columns:
        counts = df["nights_source"].value_counts()
        parsed = int(counts.get("parsed_from_card", 0))
        assumed = int(counts.get("assumed_from_search_dates", 0))
        print(f"  nights_source: {parsed}/{len(df)} parsed directly from the card, "
              f"{assumed}/{len(df)} fell back to the fixed {EXPECTED_NIGHTS}-night search window.")
        if parsed == 0:
            print("    WARNING: parse_nights() never matched on any row - every price-per-night "
                  "figure rests on the assumed fallback. This is fine ONLY because CHECKIN/CHECKOUT "
                  "are fixed for every request; if that ever changes, this becomes a real problem.")

    if "is_hotel_brand" in df.columns:
        hotel_count = df["is_hotel_brand"].sum()
        pct = 100 * hotel_count / len(df) if len(df) else 0
        print(f"  {hotel_count}/{len(df)} rows ({pct:.1f}%) flagged as hotel/aparthotel brands "
              f"- filter these out of the peer-to-peer hedonic model, or test their effect separately.")

    if "room_id" in df.columns:
        unique_ids = df["room_id"].nunique()
        print(f"  {unique_ids}/{len(df)} rows have a unique room_id "
              f"({'OK' if unique_ids == len(df) else 'WARNING: duplicates remain - check pagination'}).")

    if "price_raw" in df.columns and "price_gbp_total" in df.columns:
        # Re-derive the price straight from price_raw using the SAME logic
        # as parse_total_price, and compare it to what's recorded.
        #
        # BUG FIXED: the previous version flagged any row where a larger
        # number appeared ANYWHERE in price_raw than the recorded total.
        # That is the wrong test - for a correctly-parsed discounted row
        # like "£870 £751 total" -> 751, the struck-through £870 is SUPPOSED
        # to still be sitting in price_raw right next to the correct £751.
        # The old check fired a false "still broken" warning on every single
        # discounted listing (681/866 in the last batch) even when the
        # parser was working perfectly. Re-deriving and comparing catches
        # actual parsing failures without false-alarming on legitimate
        # struck-through prices.
        def _price_mismatch(row):
            if not row["price_raw"] or row["price_gbp_total"] is None:
                return False
            re_derived = parse_total_price(row["price_raw"])
            return re_derived is None or re_derived != row["price_gbp_total"]

        suspect_rows = df.apply(_price_mismatch, axis=1).sum()
        if suspect_rows:
            print(f"  WARNING: {suspect_rows} rows' recorded price doesn't match what "
                  f"parse_total_price(price_raw) returns now - re-check these rows individually.")
        else:
            print("  OK: every row's recorded price matches parse_total_price(price_raw).")

    print("=== End validation summary ===\n")


def save_to_excel(listings, output_file):
    df = pd.DataFrame(listings)

    # De-duplicate by room_id, NOT url. Each duplicate card carries a
    # different tracking ID in its URL query string, so url-based dedup
    # let 556 duplicate rows through last time even though drop_duplicates
    # was running. room_id is the clean numeric listing ID with no
    # query string, so true duplicates now collapse correctly.
    if "room_id" in df.columns:
        before = len(df)
        df = df.drop_duplicates(subset="room_id")
        print(f"  Deduplication: {before} rows -> {len(df)} unique listings "
              f"({before - len(df)} duplicates removed)")

    validate_output(df)

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="listings", index=False)

        # Auto-fit column widths roughly
        worksheet = writer.sheets["listings"]
        for i, col in enumerate(df.columns, start=1):
            max_len = max(df[col].astype(str).map(len).max() if not df.empty else 0, len(col)) + 2
            worksheet.column_dimensions[worksheet.cell(row=1, column=i).column_letter].width = min(max_len, 50)

    print(f"Saved {len(df)} unique listings to {output_file}")


if __name__ == "__main__":
    print(f"Starting Airbnb scrape across {len(LOCATIONS)} London areas: {LOCATIONS}")
    listings = scrape_airbnb(LOCATIONS, PAGES_PER_LOCATION, CHECKIN, CHECKOUT, headless=HEADLESS)

    if listings:
        save_to_excel(listings, OUTPUT_FILE)
    else:
        print("No listings scraped. Airbnb may have changed their page structure, "
              "shown a CAPTCHA, or blocked the request. Try HEADLESS=False to watch "
              "what the browser sees.")


Starting Airbnb scrape across 16 London areas: ['Camden, London, United Kingdom', 'Hackney, London, United Kingdom', 'Westminster, London, United Kingdom', 'Croydon, London, United Kingdom', 'Ealing, London, United Kingdom', 'Greenwich, London, United Kingdom', 'Islington, London, United Kingdom', 'Tower Hamlets, London, United Kingdom', 'Southwark, London, United Kingdom', 'Lambeth, London, United Kingdom', 'Wandsworth, London, United Kingdom', 'Bromley, London, United Kingdom', 'Barnet, London, United Kingdom', 'Enfield, London, United Kingdom', 'Redbridge, London, United Kingdom', 'Sutton, London, United Kingdom']

=== Scraping: Camden, London, United Kingdom ===
  [Page 1/2] Loading: https://www.airbnb.co.uk/s/Camden--London--United-Kingdom/homes?checkin=2026-09-01&checkout=2026-09-05
    Found 36 listing cards
  [Page 2/2] PAGINATION: 'Next' button found and clicked.
    PAGINATION: previous page's cards went stale - re-render confirmed.
    PAGINATION: new listing cards detected 